In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys; sys.path.insert(0,'../codes/')
import foftools as fof
from prob_g3groupfinder import pfof_comoving, gauss
from group_purity import get_metrics_by_group
from copy import deepcopy

%matplotlib inline

In [ ]:
eco = pd.read_csv("/srv/one/zhutchen/g3groupfinder/resolve_and_eco/ECOdata_G3catalog_luminosity.csv").set_index('name')
photzdata = pd.read_csv("/srv/one/hperk4/eco_resb_photoz.csv")

In [ ]:
photzdata = photzdata[photzdata.name.str.startswith('ECO')].set_index('name')

In [ ]:
eco = pd.concat([eco,photzdata],axis=1)

In [ ]:
eco = eco[eco.absrmag<=-19.5]

In [ ]:
degradedcz = deepcopy(eco.cz.to_numpy())
zphot = deepcopy(eco.photo_z_corr.to_numpy())
zphoterr = deepcopy(eco.e_tab_corr.to_numpy())
degradedczerr = np.zeros_like(degradedcz)+35

idx = np.random.choice(np.indices(degradedcz.shape)[0], size=int(0.85*len(degradedcz)), replace=False)
degradedcz[idx] = zphot[idx]
degradedczerr[idx] = zphoterr[idx]
sel = np.isnan(degradedcz)
degradedcz[sel] = eco.cz.to_numpy()[sel]
degradedczerr[sel] = 35

In [ ]:
eco.loc[:,'degradedcz'] = degradedcz
eco.loc[:,'degradedczerr'] = degradedczerr 

In [ ]:
pfofid=pfof_comoving(eco.radeg, eco.dedeg, eco.degradedcz+0, eco.degradedczerr, 0.07*4.84, 1.1*4.484, Pth=0.01)

In [ ]:
pfofid

In [ ]:
fofid = fof.fast_fof(eco.radeg, eco.dedeg, eco.cz, 0.07, 1.1, s=4.84)

In [ ]:
eco.loc[:,'fofid'] = fofid
eco.loc[:,'pfofid'] = pfofid

In [ ]:
pur, comp = get_metrics_by_group(eco.pfofid, eco.fofid, eco.absrmag)

In [ ]:
eco.loc[:,'pur'] = pur
eco.loc[:,'comp'] = comp
eco.loc[:,'grpn'] = fof.multiplicity_function(eco.pfofid,return_by_galaxy=True)

In [ ]:
tmp = eco.groupby('pfofid').first()
tmp = tmp[tmp.grpn>1]

plt.figure()
bins=np.arange(0,1.1,0.1)
plt.hist(tmp.pur,bins=bins)
plt.hist(tmp.comp,bins=bins,histtype='step', linewidth=2)
plt.yscale('log')
plt.show()

In [ ]:
tmp[['pur','comp','grpn']]

In [ ]:
plt.figure()
plt.scatter(tmp.grpn, tmp.pur, s=15, marker='s', edgecolor='blue', facecolor='None', label='Purity')
plt.scatter(tmp.grpn, tmp.comp, s=5, color='red', label='Completeness')
plt.xlabel(r"Group $N_{\rm giants}$")
plt.ylabel("Purity or Completeness")
plt.xlim(1,300)
plt.xscale('log')
plt.legend(loc='best')
plt.title("Purity & Completeness of PFoF-Identified Giant-only Groups\nwith respect to FoF-Identified non-degraded groups")
plt.show()

print(tmp[['pur','comp']].mean())

In [ ]:
plt.scatter(degradedcz, eco.cz)
plt.xlim(1000,10000)
plt.ylim(1000,10000)

# Test group centers

In [ ]:
test = eco[eco.pfofid==14]

In [ ]:
test[['cz','degradedcz','degradedczerr']]

In [ ]:
def gauss_vectorized(x, mu, sigma):
    """
    Gaussian function.
    Arguments:
        x - dynamic variable
        mu - centroid of distribution
        sigma - standard error of distribution
    Returns:
        PDF value evaluated at `x`
    """
    return 1/(np.sqrt(2*np.pi) * sigma) * np.exp(-1 * 0.5 * ((x-mu)/sigma) * ((x-mu)/sigma))

In [ ]:
plt.figure()

gauss_all = np.zeros(int(1e5))
for ii in range(0,len(test)):
    zz = np.linspace(0,2,int(1e5))
    gauss = gauss_vectorized(zz, np.float64(test.iloc[ii].degradedcz/3e5), np.float64(test.iloc[ii].degradedczerr/3e5))
    gauss_all+=gauss
    plt.plot(zz,gauss)
    plt.xlim(0.015,0.025)


plt.plot(zz,gauss_all,'k--', label='Group $z$ Distribution')
plt.yscale('log')
plt.ylim(1e-3,1e6)
plt.xlim(-0.01,0.13)
plt.ylabel("probability [arbitrary units]")
plt.xlabel('redshift')
plt.legend(loc='best')
plt.show()

In [ ]:
from scipy.integrate import trapezoid

In [ ]:
#gauss_all /= trapezoid(gauss_all,zz)

In [ ]:
trapezoid(gauss_all,zz)

In [ ]:
plt.plot(zz,gauss_all)
plt.xlim(0.01,0.03)
plt.ylabel("probability")
plt.xlabel("redshift")
plt.title("Coma Group $z$ Distribution")
plt.show()

In [ ]:
yy = np.cumsum(gauss_all)
plt.plot(zz,yy)
plt.xlim(0.001,0.05)

zcen = zz[np.argmin(np.abs(yy-yy.max()/2))]
print(zcen)
plt.axvline(zcen,color='k', label='Median')
plt.title("Coma $z$ Distribution CDF")
plt.legend(loc='best')
plt.xlabel("redshift")
plt.show()

In [ ]:
plt.plot(zz,gauss_all)
plt.xlim(0.01,0.03)
plt.axvline(zcen,color='k', label='Center Redshift')
plt.ylabel("probability")
plt.xlabel("redshift")
plt.title("Coma Group $z$ Distribution")
plt.legend(loc='best')
plt.show()